[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/02_evaluation/02_proyecto_libreria_evaluacion.ipynb)

# 🛠️ Proyecto 02 · `cinematch_eval`: tu propia librería de evaluación, validada contra `ranx`

| | |
|---|---|
| **Nivel** | 🟢 Básico → 🟡 Intermedio |
| **Duración estimada** | 3 h |
| **GPU** | No necesaria. ≈ 0 unidades de Colab |
| **Prerrequisitos** | Lección 02, proyecto 01 (`cinematch_data.py`) |

## 🎬 Contexto de negocio
En CineMatch ya hay tres científicos de datos probando modelos y **cada uno reporta números distintos para el mismo modelo**: uno promedia solo sobre usuarios con aciertos, otro usa 100 negativos muestreados, otro divide AP por $K$. La Head of ML te pide **la librería de evaluación oficial del equipo**: métricas exactas (verificadas contra una referencia externa), métricas *beyond-accuracy*, intervalos de confianza y tests pareados, con una interfaz única que usarán **todos los módulos siguientes del curso**:

```python
from cinematch_eval import evaluate, evaluate_per_user, bootstrap_ci, paired_bootstrap_test
res = evaluate(recs, test, k=10, train=train, items=items)
```

## 📦 Dataset
CineMatch (MovieLens-100K o 1M) con el pipeline del proyecto 01: `cinematch_data.prepare_cinematch(...)` → train / val / test temporales, positivo = rating ≥ 4.

## ✅ Entregables y rúbrica
| # | Entregable | Criterio |
|---|---|---|
| 1 | Métricas por usuario | P, R, HR, MRR, AP, NDCG (binaria y graduada, ganancia lineal y $2^r-1$) — pasan los tests unitarios |
| 2 | `evaluate_per_user` / `evaluate` | promedian sobre **todos** los usuarios de test (sin recomendaciones = 0) |
| 3 | Validación externa | diferencia máxima con `ranx` < $10^-9$ en todas las métricas y en 3+ "runs" distintos |
| 4 | Beyond-accuracy | cobertura, Gini, novedad, ILD, miscalibración (KL de Steck) |
| 5 | Métricas de *scoring* | RMSE, MAE, AUC (Mann-Whitney) y GAUC; AUC coincide con `sklearn` |
| 6 | Estadística | `bootstrap_ci` y `paired_bootstrap_test`; p-valor coherente con el t-test pareado |
| 7 | Informe | tabla de baselines de CineMatch con IC 95 % y test pareado **Item-kNN vs Popularidad** |
| 8 | `cinematch_eval.py` | guardado con `%%writefile`, importable, pasa todos los tests |

In [ ]:
!pip install -q ranx pandas pyarrow scipy scikit-learn matplotlib

In [ ]:
import importlib
import types

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)
SCALE = "small"
SIZE = "100k" if SCALE == "small" else "1m"
K = 10
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
#@title 🔧 Utilidades del curso: cinematch_data.py { display-mode: "form" }
# Escribe los módulos reutilizables del curso en el directorio actual.
# Última versión en GitHub (alternativa):
#   !wget -q https://raw.githubusercontent.com/padillacm/IA-practice/master/recsys-course/01_data/cinematch_data.py
_SOURCES = {}
_SOURCES['cinematch_data.py'] = '"""cinematch_data — pipeline de datos de CineMatch (módulo 01 del curso).\n\nInterfaz estable que reutilizan los módulos posteriores:\n\n    ratings, items = load_movielens("100k")      # o "1m", "latest-small", "25m", "32m"\n    users = load_users("100k")                    # demografía (solo 100k y 1m; si no, None)\n    inter = to_implicit(ratings, threshold=4.0)   # feedback implícito (rating >= 4)\n    inter = filter_k_core(inter, min_user=5, min_item=5)\n    train, val, test = temporal_split(inter, val_frac=0.1, test_frac=0.1)\n    train, test = leave_one_out_split(inter)      # último ítem de cada usuario a test\n    train, test = random_split(inter, test_frac=0.2)\n    X, u2i, i2i = to_csr(train)                   # matriz usuario×ítem scipy.sparse\n    recs = recommend_popular(train, users=test.user_id.unique(), k=10)\n    # recs: dict[user_id -> list[item_id]] ordenado de más a menos recomendado\n\nConvenciones de columnas: user_id (int), item_id (int), rating (float),\ntimestamp (int, segundos Unix). items: item_id, title, year, genres (str "A|B").\n\nSi la descarga de GroupLens falla (sin red), `load_movielens` genera un dataset\nsintético con el mismo esquema (`make_synthetic_movielens`) y avisa.\n"""\nfrom __future__ import annotations\n\nimport io\nimport urllib.request\nimport warnings\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nBASE_URL = "https://files.grouplens.org/datasets/movielens"\nSIZES = {\n    "100k": "ml-100k",\n    "1m": "ml-1m",\n    "latest-small": "ml-latest-small",\n    "25m": "ml-25m",\n    "32m": "ml-32m",\n}\nGENRES_100K = [\n    "unknown", "Action", "Adventure", "Animation", "Children\'s", "Comedy", "Crime",\n    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",\n    "Romance", "Sci-Fi", "Thriller", "War", "Western",\n]\n\n\n# ----------------------------------------------------------------------------\n# Descarga y carga\n# ----------------------------------------------------------------------------\ndef download_movielens(size: str = "100k", data_dir: str | Path = "data") -> Path:\n    """Descarga y descomprime MovieLens desde GroupLens. Devuelve la carpeta."""\n    if size not in SIZES:\n        raise ValueError(f"size debe ser uno de {list(SIZES)}")\n    name = SIZES[size]\n    data_dir = Path(data_dir)\n    folder = data_dir / name\n    if folder.exists():\n        return folder\n    data_dir.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE_URL}/{name}.zip"\n    print(f"Descargando {url} ...")\n    with urllib.request.urlopen(url, timeout=60) as resp:\n        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(data_dir)\n    return folder\n\n\ndef _split_title_year(titles: pd.Series) -> tuple[pd.Series, pd.Series]:\n    year = titles.str.extract(r"\\((\\d{4})\\)\\s*$")[0].astype("float")\n    clean = titles.str.replace(r"\\s*\\(\\d{4}\\)\\s*$", "", regex=True)\n    return clean, year\n\n\ndef _read_folder(size: str, folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    if size == "100k":\n        ratings = pd.read_csv(folder / "u.data", sep="\\t",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        cols = ["item_id", "title", "release_date", "video_release_date", "url"] + GENRES_100K\n        raw = pd.read_csv(folder / "u.item", sep="|", names=cols, encoding="latin-1")\n        flags = raw[GENRES_100K].values.astype(bool)\n        genres = ["|".join(g for g, f in zip(GENRES_100K, row) if f) or "unknown" for row in flags]\n        items = pd.DataFrame({"item_id": raw["item_id"], "title": raw["title"].fillna(""),\n                              "genres": genres})\n    elif size == "1m":\n        ratings = pd.read_csv(folder / "ratings.dat", sep="::", engine="python",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        items = pd.read_csv(folder / "movies.dat", sep="::", engine="python",\n                            names=["item_id", "title", "genres"], encoding="latin-1")\n    else:\n        ratings = pd.read_csv(folder / "ratings.csv").rename(\n            columns={"userId": "user_id", "movieId": "item_id"})\n        items = pd.read_csv(folder / "movies.csv").rename(columns={"movieId": "item_id"})\n    items["title"], items["year"] = _split_title_year(items["title"].astype(str))\n    return ratings, items[["item_id", "title", "year", "genres"]]\n\n\ndef load_movielens(size: str = "100k", data_dir: str | Path = "data",\n                   synthetic_fallback: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Devuelve (ratings, items) con el esquema estándar de CineMatch.\n\n    ratings: user_id, item_id, rating (float32), timestamp (int64), ordenado por tiempo.\n    items:   item_id, title, year, genres ("Action|Comedy").\n    """\n    try:\n        folder = download_movielens(size, data_dir)\n        ratings, items = _read_folder(size, folder)\n    except Exception as e:  # sin red, URL caída, etc.\n        if not synthetic_fallback:\n            raise\n        warnings.warn(f"No se pudo descargar MovieLens-{size} ({e!r}). "\n                      "Usando datos SINTÉTICOS con el mismo esquema.")\n        ratings, items = make_synthetic_movielens()\n    ratings = ratings.astype({"user_id": "int64", "item_id": "int64",\n                              "rating": "float32", "timestamp": "int64"})\n    ratings = ratings.sort_values(["timestamp", "user_id", "item_id"], kind="stable")\n    return ratings.reset_index(drop=True), items.reset_index(drop=True)\n\n\ndef load_users(size: str = "100k", data_dir: str | Path = "data") -> pd.DataFrame | None:\n    """Demografía de usuarios (user_id, age, gender, occupation). Solo 100k y 1m."""\n    try:\n        folder = download_movielens(size, data_dir)\n    except Exception:\n        return None\n    if size == "100k":\n        return pd.read_csv(folder / "u.user", sep="|",\n                           names=["user_id", "age", "gender", "occupation", "zip"])\n    if size == "1m":\n        return pd.read_csv(folder / "users.dat", sep="::", engine="python",\n                           names=["user_id", "gender", "age", "occupation", "zip"])\n    return None\n\n\ndef make_synthetic_movielens(n_users: int = 943, n_items: int = 1682,\n                             n_ratings: int = 100_000, seed: int = 42\n                             ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Dataset sintético con forma de MovieLens: long tail, actividad sesgada,\n    gustos latentes por género y deriva temporal (los ítems nuevos van llegando)."""\n    rng = np.random.default_rng(seed)\n    genres = GENRES_100K[1:]\n    t0, t1 = 874_724_710, 893_286_638  # mismo rango temporal que ML-100K\n    # Ítems: género principal, popularidad intrínseca (Zipf) y fecha de llegada\n    item_genre = rng.integers(0, len(genres), n_items)\n    item_pop = 1.0 / np.arange(1, n_items + 1) ** 1.1\n    rng.shuffle(item_pop)\n    item_arrival = t0 + (t1 - t0) * np.where(rng.random(n_items) < 0.5, 0.0,\n                                             rng.uniform(0, 0.95, n_items))  # catálogo inicial + estrenos\n    # Usuarios: actividad log-normal, afinidad por géneros y fecha de alta\n    activity = rng.lognormal(0, 1.0, n_users)\n    activity = np.maximum(20, activity / activity.sum() * n_ratings).astype(int)\n    taste = rng.dirichlet(np.full(len(genres), 0.3), n_users)\n    user_start = t0 + (t1 - t0) * rng.uniform(0, 0.9, n_users)\n    item_quality = rng.normal(0, 0.45, n_items)   # unas películas son mejores que otras\n    user_bias = rng.normal(0, 0.35, n_users)      # hay usuarios generosos y exigentes\n    rows = []\n    for u in range(n_users):\n        start = user_start[u]\n        end = min(t1, start + rng.uniform(1, 120) * 86_400)\n        avail = item_arrival <= end\n        n = int(min(activity[u], avail.sum() // 2))\n        ts = np.sort(rng.uniform(start, end, n))\n        age_days = np.maximum(0, end - item_arrival) / 86_400\n        hype = np.where(item_arrival > t0, 1 + 30 * np.exp(-age_days / 30), 1.0)  # los estrenos tienen su momento\n        w = item_pop * hype * (0.15 + taste[u][item_genre]) * avail\n        chosen = rng.choice(n_items, size=n, replace=False, p=w / w.sum())\n        ts = np.maximum(ts, item_arrival[chosen] + 3600)\n        aff = taste[u][item_genre[chosen]] * len(genres)\n        r = np.clip(np.round(3.2 + 0.6 * np.log1p(aff) + item_quality[chosen] + user_bias[u]\n                             + rng.normal(0, 0.7, n)), 1, 5)\n        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": chosen + 1,\n                                  "rating": r, "timestamp": ts.astype(np.int64)}))\n    ratings = pd.concat(rows, ignore_index=True)\n    second = rng.integers(0, len(genres), n_items)\n    items = pd.DataFrame({\n        "item_id": np.arange(1, n_items + 1),\n        "title": [f"Película sintética {i}" for i in range(1, n_items + 1)],\n        "year": (1930 + 66 * rng.beta(4, 1.5, n_items)).round(),\n        "genres": [genres[a] if a == b else f"{genres[a]}|{genres[b]}"\n                   for a, b in zip(item_genre, second)],\n    })\n    return ratings, items\n\n\n# ----------------------------------------------------------------------------\n# Limpieza y señales\n# ----------------------------------------------------------------------------\ndef to_implicit(ratings: pd.DataFrame, threshold: float = 4.0) -> pd.DataFrame:\n    """Convierte ratings explícitos en interacciones implícitas positivas\n    (rating >= threshold). Mantiene la columna rating por si se quiere ponderar."""\n    out = ratings[ratings["rating"] >= threshold].copy()\n    return out.reset_index(drop=True)\n\n\ndef dedup_interactions(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:\n    """Elimina pares (usuario, ítem) duplicados quedándose con el primero/último en el tiempo."""\n    return (df.sort_values("timestamp", kind="stable")\n              .drop_duplicates(["user_id", "item_id"], keep=keep)\n              .reset_index(drop=True))\n\n\ndef filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5,\n                  max_iter: int = 100) -> pd.DataFrame:\n    """k-core iterativo: repite hasta que todo usuario tenga >= min_user\n    interacciones y todo ítem >= min_item (filtrar uno puede romper el otro)."""\n    for _ in range(max_iter):\n        n0 = len(df)\n        ic = df["item_id"].map(df["item_id"].value_counts())\n        df = df[ic >= min_item]\n        uc = df["user_id"].map(df["user_id"].value_counts())\n        df = df[uc >= min_user]\n        if len(df) == n0:\n            break\n    return df.reset_index(drop=True)\n\n\n# ----------------------------------------------------------------------------\n# Splits\n# ----------------------------------------------------------------------------\ndef _drop_cold(train: pd.DataFrame, other: pd.DataFrame, users: bool, items: bool) -> pd.DataFrame:\n    if users:\n        other = other[other["user_id"].isin(train["user_id"].unique())]\n    if items:\n        other = other[other["item_id"].isin(train["item_id"].unique())]\n    return other.reset_index(drop=True)\n\n\ndef random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,\n                 drop_cold: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Split aleatorio de interacciones (¡filtra el futuro al train! solo como contraejemplo)."""\n    mask = np.random.default_rng(seed).random(len(df)) < test_frac\n    train, test = df[~mask].reset_index(drop=True), df[mask]\n    return train, _drop_cold(train, test, drop_cold, drop_cold)\n\n\ndef leave_one_out_split(df: pd.DataFrame, n_test: int = 1, min_train: int = 1\n                        ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Las últimas n_test interacciones (en el tiempo) de cada usuario van a test.\n    Ojo: no hay un corte temporal global → el train de un usuario puede contener\n    interacciones posteriores al test de otro (leakage temporal entre usuarios)."""\n    df = df.sort_values(["user_id", "timestamp"], kind="stable")\n    rank_from_end = df.groupby("user_id").cumcount(ascending=False)\n    n_user = df.groupby("user_id")["item_id"].transform("size")\n    is_test = (rank_from_end < n_test) & (n_user >= n_test + min_train)\n    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)\n\n\ndef temporal_split(df: pd.DataFrame, val_frac: float = 0.1, test_frac: float = 0.1,\n                   drop_cold_users: bool = True, drop_cold_items: bool = True\n                   ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Split temporal GLOBAL: train = pasado, val = siguiente val_frac del tiempo\n    (por cuantiles de eventos), test = último test_frac. Es lo más parecido a\n    producción: entrenas con todo lo anterior a T y sirves después de T.\n    Devuelve (train, val, test); val puede estar vacío si val_frac == 0."""\n    ts = df["timestamp"]\n    t_test = ts.quantile(1 - test_frac)\n    t_val = ts.quantile(1 - test_frac - val_frac) if val_frac > 0 else t_test\n    train = df[ts < t_val].reset_index(drop=True)\n    val = df[(ts >= t_val) & (ts < t_test)]\n    test = df[ts >= t_test]\n    val = _drop_cold(train, val, drop_cold_users, drop_cold_items)\n    # Para test, "conocido" = visto en train o val (en producción reentrenas antes de servir)\n    seen = pd.concat([train, val])\n    test = _drop_cold(seen, test, drop_cold_users, drop_cold_items)\n    return train, val, test\n\n\n# ----------------------------------------------------------------------------\n# Matrices dispersas\n# ----------------------------------------------------------------------------\ndef build_mappings(df: pd.DataFrame) -> tuple[dict, dict]:\n    """Mapeos id original -> índice contiguo 0..n-1 (usuarios e ítems)."""\n    users = np.sort(df["user_id"].unique())\n    items = np.sort(df["item_id"].unique())\n    return ({u: i for i, u in enumerate(users)}, {it: j for j, it in enumerate(items)})\n\n\ndef to_csr(df: pd.DataFrame, user2idx: dict | None = None, item2idx: dict | None = None,\n           value_col: str | None = None) -> tuple[sp.csr_matrix, dict, dict]:\n    """Matriz usuario×ítem CSR. value_col=None → 1.0 por interacción (implícito).\n    Las filas/ítems fuera del mapeo se descartan. Devuelve (X, user2idx, item2idx)."""\n    if user2idx is None or item2idx is None:\n        user2idx, item2idx = build_mappings(df)\n    r = df["user_id"].map(user2idx)\n    c = df["item_id"].map(item2idx)\n    ok = r.notna() & c.notna()\n    vals = np.ones(ok.sum(), dtype=np.float32) if value_col is None \\\n        else df.loc[ok, value_col].to_numpy(np.float32)\n    X = sp.csr_matrix((vals, (r[ok].astype(int), c[ok].astype(int))),\n                      shape=(len(user2idx), len(item2idx)))\n    X.sum_duplicates()\n    return X, user2idx, item2idx\n\n\n# ----------------------------------------------------------------------------\n# Baselines\n# ----------------------------------------------------------------------------\ndef _seen_by_user(train: pd.DataFrame) -> dict:\n    return train.groupby("user_id")["item_id"].agg(set).to_dict()\n\n\ndef _topk_excluding(ranked: np.ndarray, seen: set, k: int) -> list:\n    out = []\n    for it in ranked:\n        if it not in seen:\n            out.append(int(it))\n            if len(out) == k:\n                break\n    return out\n\n\ndef popularity_ranking(train: pd.DataFrame, window_days: float | None = None,\n                       half_life_days: float | None = None) -> np.ndarray:\n    """Ítems ordenados por popularidad. window_days: solo cuenta los últimos N días.\n    half_life_days: pondera cada evento por 0.5**(edad/half_life) (recencia suave)."""\n    df = train\n    t_max = df["timestamp"].max()\n    if window_days is not None:\n        df = df[df["timestamp"] >= t_max - window_days * 86_400]\n    if half_life_days is not None:\n        age = (t_max - df["timestamp"]) / 86_400\n        w = 0.5 ** (age / half_life_days)\n        scores = w.groupby(df["item_id"]).sum()\n    else:\n        scores = df["item_id"].value_counts()\n    return scores.sort_values(ascending=False, kind="stable").index.to_numpy()\n\n\ndef recommend_popular(train: pd.DataFrame, users, k: int = 10, exclude_seen: bool = True,\n                      window_days: float | None = None,\n                      half_life_days: float | None = None) -> dict:\n    """Top-K más populares (opcionalmente recientes) para cada usuario de `users`."""\n    ranked = popularity_ranking(train, window_days, half_life_days)\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranked, seen.get(u, set()), k) for u in users}\n\n\ndef recommend_popular_by_segment(train: pd.DataFrame, users, segment_of: dict, k: int = 10,\n                                 exclude_seen: bool = True, min_segment_events: int = 50) -> dict:\n    """Popularidad dentro del segmento del usuario (p. ej. edad, país, género favorito).\n    `segment_of`: dict user_id -> segmento. Segmentos con pocos eventos → popularidad global."""\n    seg = train["user_id"].map(segment_of)\n    global_rank = popularity_ranking(train)\n    ranks = {}\n    for s, g in train.groupby(seg):\n        if len(g) >= min_segment_events:\n            r = g["item_id"].value_counts().index.to_numpy()\n            # completar con el ranking global para no quedarnos cortos\n            ranks[s] = np.concatenate([r, global_rank[~np.isin(global_rank, r)]])\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranks.get(segment_of.get(u), global_rank), seen.get(u, set()), k)\n            for u in users}\n\n\ndef recommend_random(train: pd.DataFrame, users, k: int = 10, seed: int = 42,\n                     exclude_seen: bool = True) -> dict:\n    """Recomendador aleatorio: el suelo absoluto de cualquier comparación."""\n    rng = np.random.default_rng(seed)\n    items = train["item_id"].unique()\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(rng.permutation(items), seen.get(u, set()), k) for u in users}\n\n\n# ----------------------------------------------------------------------------\n# Persistencia\n# ----------------------------------------------------------------------------\ndef save_splits(out_dir: str | Path, **frames: pd.DataFrame) -> Path:\n    """Guarda cada DataFrame como parquet: save_splits("data/cinematch", train=..., test=...)."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for name, f in frames.items():\n        f.to_parquet(out_dir / f"{name}.parquet", index=False)\n    return out_dir\n\n\ndef load_splits(out_dir: str | Path, names=("train", "val", "test", "items")) -> dict:\n    out_dir = Path(out_dir)\n    return {n: pd.read_parquet(out_dir / f"{n}.parquet") for n in names\n            if (out_dir / f"{n}.parquet").exists()}\n\n\n# ----------------------------------------------------------------------------\n# Pipeline completo de CineMatch (una llamada)\n# ----------------------------------------------------------------------------\ndef prepare_cinematch(size: str = "100k", threshold: float | None = 4.0, min_user: int = 5,\n                      min_item: int = 5, val_frac: float = 0.1, test_frac: float = 0.1,\n                      data_dir: str | Path = "data", out_dir: str | Path | None = None) -> dict:\n    """Pipeline estándar del curso:\n    carga → dedup → implícito (rating >= threshold; None = no filtrar) → split temporal\n    global → k-core SOLO sobre train (no miramos el futuro para filtrar) → val/test\n    restringidos a usuarios e ítems conocidos en train.\n\n    Devuelve {"train", "val", "test", "items", "ratings"} (DataFrames) y, si\n    out_dir no es None, los guarda en parquet."""\n    ratings, items = load_movielens(size, data_dir)\n    inter = dedup_interactions(ratings)\n    if threshold is not None:\n        inter = to_implicit(inter, threshold)\n    train, val, test = temporal_split(inter, val_frac, test_frac,\n                                      drop_cold_users=False, drop_cold_items=False)\n    train = filter_k_core(train, min_user, min_item)\n    val = _drop_cold(train, val, True, True)\n    test = _drop_cold(train, test, True, True)\n    out = {"train": train, "val": val, "test": test, "items": items, "ratings": ratings}\n    if out_dir is not None:\n        save_splits(out_dir, train=train, val=val, test=test, items=items)\n    return out\n'
for _name, _src in _SOURCES.items():
    with open(_name, 'w', encoding='utf-8') as _fh:
        _fh.write(_src)
    print('✔ escrito', _name)

In [ ]:
import cinematch_data as cd

data = cd.prepare_cinematch(SIZE)
train, val, test, items = data["train"], data["val"], data["test"], data["items"]
train_full = pd.concat([train, val])
print({k: len(v) for k, v in data.items()})

### Recomendaciones a evaluar
Generamos las listas de 4 baselines (aleatorio, popularidad, popularidad 30 días, item-kNN por co-ocurrencia) con `cinematch_data` y una implementación mínima de item-kNN. **Tu trabajo es evaluarlas**, no construirlas.

In [ ]:
def item_knn_recs(train: pd.DataFrame, users, k: int = 100) -> dict:
    X, u2i, i2i = cd.to_csr(train); X = (X > 0).astype(np.float32)
    idx2item = np.array(sorted(i2i, key=i2i.get))
    co = (X.T @ X).toarray(); d = np.diag(co).copy()
    S = co / (np.sqrt(np.outer(d, d)) + 1e-9); np.fill_diagonal(S, 0)
    out = {}
    for u in users:
        if u not in u2i:
            continue                                   # usuario sin historial: sin recomendación (cuenta 0)
        s = np.asarray(X[u2i[u]] @ S).ravel(); s[X[u2i[u]].indices] = -np.inf
        idx = np.argpartition(-s, k)[:k]; out[u] = idx2item[idx[np.argsort(-s[idx])]].tolist()
    return out

us = test.user_id.unique()
RECS = {"Aleatorio": cd.recommend_random(train_full, us, k=100),
        "Popularidad": cd.recommend_popular(train_full, us, k=100),
        "Popularidad 30 d": cd.recommend_popular(train_full, us, k=100, window_days=30),
        "Item-kNN": item_knn_recs(train_full, us)}
{m: len(r) for m, r in RECS.items()}

---
## Parte 1 · Métricas por usuario

### TODO 1 — Seis métricas
Firma común: `(rec: list, rel: set | dict, k: int) -> float`. Convenciones obligatorias (las de `ranx`/`trec_eval`):
- `precision_at_k` divide por $k$ aunque la lista sea más corta.
- `average_precision_at_k(rec, rel, k, denominator="rel")` divide por $|\mathcal{R}|$; con `denominator="min"` por $\min(k, |\mathcal{R}|)$.
- `ndcg_at_k(rec, rel, k, exp_gain=False)`: si `rel` es un `dict` ítem → relevancia graduada; ganancia lineal $r$ o, con `exp_gain=True`, $2^r - 1$.

In [ ]:
def precision_at_k(rec, rel, k):
    raise NotImplementedError  # TODO

def recall_at_k(rec, rel, k):
    raise NotImplementedError  # TODO

def hit_rate_at_k(rec, rel, k):
    raise NotImplementedError  # TODO

def mrr_at_k(rec, rel, k):
    raise NotImplementedError  # TODO

def average_precision_at_k(rec, rel, k, denominator="rel"):
    raise NotImplementedError  # TODO

def ndcg_at_k(rec, rel, k, exp_gain=False):
    raise NotImplementedError  # TODO

### Tests unitarios (calculados a mano)
Lista `[1, 2, 3, 4, 5]`, relevantes `{2, 5, 9}`, $k=5$: P = 0,4 · R = 2/3 · HR = 1 · MRR = 0,5 · AP = (1/2 + 2/5)/3 = 0,3 · NDCG = $\frac{1/\log_2 3 + 1/\log_2 6}{1 + 1/\log_2 3 + 1/\log_2 4}$ = 0,47762. La función recibe un "espacio de nombres" con tus funciones para poder reutilizarla luego con el módulo.

In [ ]:
def tests_unitarios(m) -> None:
    rec, rel = [1, 2, 3, 4, 5], {2, 5, 9}
    assert np.isclose(m.precision_at_k(rec, rel, 5), 0.4)
    assert np.isclose(m.precision_at_k([2], rel, 5), 0.2), "precision divide por k"
    assert np.isclose(m.recall_at_k(rec, rel, 5), 2 / 3)
    assert m.hit_rate_at_k(rec, rel, 5) == 1.0 and m.hit_rate_at_k(rec, rel, 1) == 0.0
    assert np.isclose(m.mrr_at_k(rec, rel, 5), 0.5) and m.mrr_at_k(rec, rel, 1) == 0.0
    assert np.isclose(m.average_precision_at_k(rec, rel, 5), 0.3)
    assert np.isclose(m.average_precision_at_k([2, 5], rel, 2, "min"), 1.0)
    assert np.isclose(m.ndcg_at_k(rec, rel, 5), 0.4776237035032179)
    assert np.isclose(m.ndcg_at_k([9, 2, 5], rel, 3), 1.0), "lista ideal → NDCG = 1"
    graded = {1: 1, 3: 3, 7: 2}
    assert np.isclose(m.ndcg_at_k([1, 2, 3], graded, 3), 0.5250049893849101)
    assert np.isclose(m.ndcg_at_k([1, 2, 3], graded, 3, exp_gain=True), 0.47909091485969846)
    assert m.recall_at_k([], rel, 5) == 0.0 and m.ndcg_at_k([], rel, 5) == 0.0
    print("✔ tests unitarios superados")

tests_unitarios(types.SimpleNamespace(**globals()))

### TODO 2 — Agregación
- `evaluate_per_user(recs, test, k, metrics, rel_col=None, exp_gain=False, ap_denominator="rel") -> DataFrame` con índice `user_id` y columnas `"ndcg@10"`, etc. **Todos** los usuarios de `test`; si `rel_col` (p. ej. `"rating"`) se pasa, NDCG usa relevancia graduada y el resto usa como relevantes los de relevancia > 0.
- `evaluate(recs, test, k, metrics, rel_col=None, train=None, items=None, **kw) -> dict` con las medias, y si se pasa `train` (e `items`) añade las métricas *beyond-accuracy* del TODO 4.

In [ ]:
DEFAULT_METRICS = ("precision", "recall", "hit_rate", "mrr", "map", "ndcg")

def evaluate_per_user(recs, test, k=10, metrics=DEFAULT_METRICS, rel_col=None, exp_gain=False, ap_denominator="rel"):
    raise NotImplementedError  # TODO

def evaluate(recs, test, k=10, metrics=DEFAULT_METRICS, rel_col=None, train=None, items=None, **kw):
    raise NotImplementedError  # TODO

### TODO 3 — Validación contra `ranx`
Implementa `to_ranx(recs, test, rel_col=None) -> (Qrels, Run)` (puntuación decreciente con la posición; un usuario sin recomendaciones necesita una entrada "centinela" porque `ranx` no admite consultas vacías) y `validar_contra_ranx(m)`, que para cada run de `RECS` y para $k \in \{5, 10, 20\}$ compruebe las 6 métricas binarias **y** NDCG graduado con `rel_col="rating"` (lineal → `ndcg`, exponencial → `ndcg_burges`). Diferencia máxima < $10^{-9}$.

In [ ]:
from ranx import Qrels, Run, compare
from ranx import evaluate as rx_evaluate

def to_ranx(recs, test, rel_col=None):
    raise NotImplementedError  # TODO

def validar_contra_ranx(m) -> None:
    raise NotImplementedError  # TODO

validar_contra_ranx(types.SimpleNamespace(**globals()))

---
## Parte 2 · Más allá de la precisión, *scoring* y estadística

### TODO 4 — Beyond-accuracy
- `catalog_coverage(recs, catalog, k)`, `gini_index(recs, catalog, k)` (de la **exposición**: cuántas veces aparece cada ítem del catálogo en los top-k).
- `novelty(recs, train, k)`: media de $-\log_2 p(i)$, $p(i)$ = fracción de usuarios de train que vieron $i$.
- `intra_list_diversity(recs, items, k)`: media de $1-\cos$ entre pares, con vectores multi-hot de `genres`.
- `miscalibration_kl(recs, train, items, k, alpha=0.01)`: Steck (2018). $p(g\mid u)$ = distribución de géneros del historial (cada película reparte 1 entre sus géneros), $q(g\mid u)$ = la de la lista; $\text{KL}(p \,\|\, \tilde q)$ con $\tilde q = (1-\alpha) q + \alpha p$.

### TODO 5 — Métricas de *scoring*
`rmse`, `mae`, `auc_score(labels, scores)` (Mann-Whitney con rangos promedio para empates; compárala con `sklearn.metrics.roc_auc_score`) y `gauc(df, user_col, label_col, score_col)` (AUC por usuario ponderada por nº de filas; ignora usuarios con una sola clase).

### TODO 6 — Estadística
`bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=42) -> (media, lo, hi)` y `paired_bootstrap_test(a, b, n_boot=10_000, seed=42) -> dict(diff, ci_low, ci_high, p_value)` (p-valor bilateral centrando la distribución bootstrap de la diferencia en 0).

In [ ]:
def catalog_coverage(recs, catalog, k=10):         raise NotImplementedError  # TODO
def gini_index(recs, catalog, k=10):               raise NotImplementedError  # TODO
def novelty(recs, train, k=10):                    raise NotImplementedError  # TODO
def intra_list_diversity(recs, items, k=10):       raise NotImplementedError  # TODO
def miscalibration_kl(recs, train, items, k=10, alpha=0.01): raise NotImplementedError  # TODO

def rmse(y_true, y_pred):                          raise NotImplementedError  # TODO
def mae(y_true, y_pred):                           raise NotImplementedError  # TODO
def auc_score(labels, scores):                     raise NotImplementedError  # TODO
def gauc(df, user_col="user_id", label_col="label", score_col="score"): raise NotImplementedError  # TODO

def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=42):        raise NotImplementedError  # TODO
def paired_bootstrap_test(a, b, n_boot=10_000, seed=42):           raise NotImplementedError  # TODO

In [ ]:
def tests_parte2(m) -> None:
    recs = {1: [1, 2], 2: [1, 3]}
    assert np.isclose(m.catalog_coverage(recs, [1, 2, 3, 4], 2), 0.75)
    assert np.isclose(m.gini_index({1: [1], 2: [2]}, [1, 2], 1), 0.0)
    tr = pd.DataFrame({"user_id": [1, 2, 2, 3], "item_id": [1, 1, 2, 3]})
    assert np.isclose(m.novelty({9: [1]}, tr, 1), -np.log2(2 / 3))
    it = pd.DataFrame({"item_id": [1, 2, 3], "genres": ["A", "A", "B"]})
    assert np.isclose(m.intra_list_diversity({1: [1, 2]}, it, 2), 0.0)
    assert np.isclose(m.intra_list_diversity({1: [1, 3]}, it, 2), 1.0)
    assert m.miscalibration_kl({1: [1]}, pd.DataFrame({"user_id": [1], "item_id": [2]}), it, 1) < 1e-9
    from sklearn.metrics import roc_auc_score
    y = rng.integers(0, 2, 500); s = rng.normal(size=500) + y; s[:50] = np.round(s[:50])
    assert np.isclose(m.auc_score(y, s), roc_auc_score(y, s))
    df = pd.DataFrame({"user_id": [1, 1, 1, 2, 2, 3], "label": [1, 0, 0, 1, 0, 1], "score": [.9, .1, .5, .2, .8, .3]})
    assert np.isclose(m.gauc(df), (3 * 1.0 + 2 * 0.0) / 5)
    assert np.isclose(m.rmse([1, 2], [1, 4]), np.sqrt(2)) and np.isclose(m.mae([1, 2], [1, 4]), 1.0)
    v = rng.normal(0.1, 0.05, 400)
    mean, lo, hi = m.bootstrap_ci(v)
    assert lo < mean < hi and np.isclose(mean, v.mean())
    a = v + rng.normal(0.01, 0.02, 400)
    r = m.paired_bootstrap_test(a, v)
    assert r["p_value"] < 0.01 and r["ci_low"] > 0
    e = rng.normal(0, 0.02, 400); e -= e.mean()                       # diferencia media exactamente 0
    assert m.paired_bootstrap_test(v, v + e)["p_value"] > 0.5
    print("✔ tests de la parte 2 superados")

tests_parte2(types.SimpleNamespace(**globals()))

---
## Parte 3 · Informe y empaquetado

### TODO 7 — Informe de baselines de CineMatch
Con **tus** funciones: tabla con `evaluate(..., k=10, train=train_full, items=items)` para los 4 modelos; NDCG@10 con IC 95 % (gráfico de barras de error); test pareado Item-kNN vs Popularidad (tu bootstrap y `scipy.stats.ttest_rel`) y `ranx.compare`. Conclusión en 3 frases.

In [ ]:
# TODO

### TODO 8 — Guarda tu librería
Copia **todas** tus funciones (con sus imports y docstrings) en la celda de abajo, ejecútala y verifica que el módulo pasa los tres bloques de tests.

In [ ]:
%%writefile cinematch_eval.py
"""cinematch_eval — evaluador offline de CineMatch."""
# TODO: pega aquí tus imports y funciones

In [ ]:
import cinematch_eval
importlib.reload(cinematch_eval)
tests_unitarios(cinematch_eval); tests_parte2(cinematch_eval); validar_contra_ranx(cinematch_eval)

---
## ⛔ SPOILER — Solución de referencia

Intenta resolverlo primero. La solución de referencia es el módulo canónico del curso, `cinematch_eval.py`: lo escribimos en disco por partes, lo importamos, pasamos **todos los tests** y generamos el informe.

In [ ]:
%%writefile cinematch_eval.py
"""cinematch_eval — evaluador offline de CineMatch (módulo 02 del curso).

Métricas implementadas desde cero y verificadas contra `ranx`.

Formato de entrada (convención de todo el curso):
    recs : dict[user_id -> list[item_id]]  ordenada (posición 0 = la mejor)
    test : pd.DataFrame con columnas user_id, item_id (+ opcional una columna de
           relevancia graduada, p. ej. "rating", vía rel_col)

Uso:
    from cinematch_eval import evaluate, evaluate_per_user, bootstrap_ci
    res = evaluate(recs, test, k=10)                       # dict métrica -> media
    res = evaluate(recs, test, k=10, train=train, items=items)  # + beyond-accuracy
    per_user = evaluate_per_user(recs, test, k=10)          # DataFrame (para tests)

Convenciones (¡importan al comparar con papers!):
  * Se promedia sobre TODOS los usuarios de `test`; un usuario sin recomendaciones
    cuenta como 0 (no se le "olvida").
  * precision@k divide por k aunque la lista sea más corta.
  * AP@k divide por |relevantes| (convención de ranx/trec_eval);
    con ap_denominator="min" divide por min(k, |relevantes|).
  * NDCG usa ganancia lineal rel/log2(pos+1) (ranx "ndcg"); con exp_gain=True usa
    2^rel - 1 (ranx "ndcg_burges"). Con relevancia binaria ambas coinciden.
  * Evaluación full-ranking (contra todo el catálogo). Nada de sampled metrics.
"""
from __future__ import annotations

from collections import Counter

import numpy as np
import pandas as pd

DEFAULT_METRICS = ("precision", "recall", "hit_rate", "mrr", "map", "ndcg")


# ----------------------------------------------------------------------------
# Métricas por usuario (una lista recomendada vs un conjunto relevante)
# ----------------------------------------------------------------------------
def precision_at_k(rec: list, rel: set, k: int) -> float:
    return sum(1 for i in rec[:k] if i in rel) / k


def recall_at_k(rec: list, rel: set, k: int) -> float:
    if not rel:
        return 0.0
    return sum(1 for i in rec[:k] if i in rel) / len(rel)


def hit_rate_at_k(rec: list, rel: set, k: int) -> float:
    return float(any(i in rel for i in rec[:k]))


def mrr_at_k(rec: list, rel: set, k: int) -> float:
    for pos, i in enumerate(rec[:k], start=1):
        if i in rel:
            return 1.0 / pos
    return 0.0

In [ ]:
%%writefile -a cinematch_eval.py


def average_precision_at_k(rec: list, rel: set, k: int, denominator: str = "rel") -> float:
    if not rel:
        return 0.0
    hits, s = 0, 0.0
    for pos, i in enumerate(rec[:k], start=1):
        if i in rel:
            hits += 1
            s += hits / pos
    denom = len(rel) if denominator == "rel" else min(k, len(rel))
    return s / denom


def ndcg_at_k(rec: list, rel, k: int, exp_gain: bool = False) -> float:
    """rel: set (binaria) o dict item -> relevancia graduada."""
    gains = rel if isinstance(rel, dict) else {i: 1.0 for i in rel}
    if not gains:
        return 0.0
    g = (lambda x: 2.0 ** x - 1.0) if exp_gain else (lambda x: float(x))
    dcg = sum(g(gains.get(i, 0.0)) / np.log2(pos + 1) for pos, i in enumerate(rec[:k], start=1))
    ideal = sorted(gains.values(), reverse=True)[:k]
    idcg = sum(g(x) / np.log2(pos + 1) for pos, x in enumerate(ideal, start=1))
    return dcg / idcg if idcg > 0 else 0.0


_METRIC_FNS = {
    "precision": precision_at_k,
    "recall": recall_at_k,
    "hit_rate": hit_rate_at_k,
    "mrr": mrr_at_k,
    "map": average_precision_at_k,
    "ndcg": ndcg_at_k,
}


# ----------------------------------------------------------------------------
# Métricas de predicción de rating (explícito)
# ----------------------------------------------------------------------------
def rmse(y_true, y_pred) -> float:
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred) -> float:
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.mean(np.abs(y_true - y_pred)))


def auc_score(labels, scores) -> float:
    """AUC = P(score positivo > score negativo), empates cuentan 0.5 (Mann-Whitney)."""
    labels, scores = np.asarray(labels).astype(bool), np.asarray(scores, float)
    n_pos, n_neg = labels.sum(), (~labels).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(scores).rank(method="average").to_numpy()
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def gauc(df: pd.DataFrame, user_col: str = "user_id", label_col: str = "label",
         score_col: str = "score") -> float:
    """Group AUC (Alibaba): AUC por usuario ponderada por nº de impresiones.
    Usuarios con solo positivos o solo negativos se ignoran."""
    num, den = 0.0, 0
    for _, g in df.groupby(user_col):
        a = auc_score(g[label_col].to_numpy(), g[score_col].to_numpy())
        if not np.isnan(a):
            num += a * len(g)
            den += len(g)
    return num / den if den else float("nan")

In [ ]:
%%writefile -a cinematch_eval.py


# ----------------------------------------------------------------------------
# Agregación
# ----------------------------------------------------------------------------
def _relevance(test: pd.DataFrame, rel_col: str | None) -> dict:
    if rel_col is None:
        return test.groupby("user_id")["item_id"].agg(set).to_dict()
    return {u: dict(zip(g["item_id"], g[rel_col])) for u, g in test.groupby("user_id")}


def evaluate_per_user(recs: dict, test: pd.DataFrame, k: int = 10,
                      metrics=DEFAULT_METRICS, rel_col: str | None = None,
                      exp_gain: bool = False, ap_denominator: str = "rel") -> pd.DataFrame:
    """DataFrame (índice user_id) con una columna por métrica, p. ej. 'ndcg@10'.
    rel_col: columna de relevancia graduada (solo afecta a NDCG; el resto usa
    como relevantes los ítems con rel > 0)."""
    rel = _relevance(test, rel_col)
    rows = {}
    for u, r in rel.items():
        rec = list(recs.get(u, []))
        rel_set = {i for i, v in r.items() if v > 0} if isinstance(r, dict) else r
        row = {}
        for m in metrics:
            if m == "ndcg":
                row[f"ndcg@{k}"] = ndcg_at_k(rec, r, k, exp_gain)
            elif m == "map":
                row[f"map@{k}"] = average_precision_at_k(rec, rel_set, k, ap_denominator)
            else:
                row[f"{m}@{k}"] = _METRIC_FNS[m](rec, rel_set, k)
        rows[u] = row
    out = pd.DataFrame.from_dict(rows, orient="index")
    out.index.name = "user_id"
    return out


# ----------------------------------------------------------------------------
# Beyond-accuracy
# ----------------------------------------------------------------------------
def catalog_coverage(recs: dict, catalog, k: int = 10) -> float:
    """Fracción del catálogo que aparece en al menos una lista top-k."""
    shown = {i for rec in recs.values() for i in rec[:k]}
    catalog = set(catalog)
    return len(shown & catalog) / len(catalog)


def gini_index(recs: dict, catalog, k: int = 10) -> float:
    """Gini de la exposición de ítems (0 = exposición uniforme, 1 = todo a un ítem)."""
    cnt = Counter(i for rec in recs.values() for i in rec[:k])
    x = np.sort(np.array([cnt.get(i, 0) for i in set(catalog)], float))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))


def novelty(recs: dict, train: pd.DataFrame, k: int = 10) -> float:
    """Autoinformación media -log2 p(i), p(i) = fracción de usuarios de train que vieron i.
    Más alta = recomiendas cosas menos conocidas."""
    n_users = train["user_id"].nunique()
    p = train.groupby("item_id")["user_id"].nunique() / n_users
    vals = [-np.log2(p.get(i, 1.0 / n_users)) for rec in recs.values() for i in rec[:k]]
    return float(np.mean(vals)) if vals else 0.0


def _genre_matrix(items: pd.DataFrame) -> pd.DataFrame:
    return items.set_index("item_id")["genres"].str.get_dummies(sep="|").astype(float)

In [ ]:
%%writefile -a cinematch_eval.py


def intra_list_diversity(recs: dict, items: pd.DataFrame, k: int = 10) -> float:
    """ILD: media de (1 - coseno) entre pares de ítems de cada lista (vectores de género)."""
    G = _genre_matrix(items)
    G = G.div(np.linalg.norm(G.values, axis=1).clip(1e-12), axis=0)
    vals = []
    for rec in recs.values():
        rec = [i for i in rec[:k] if i in G.index]
        if len(rec) < 2:
            continue
        V = G.loc[rec].values
        S = V @ V.T
        n = len(rec)
        vals.append(1 - (S.sum() - np.trace(S)) / (n * (n - 1)))
    return float(np.mean(vals)) if vals else 0.0


def miscalibration_kl(recs: dict, train: pd.DataFrame, items: pd.DataFrame,
                      k: int = 10, alpha: float = 0.01) -> float:
    """Miscalibración de Steck (2018): KL(p || q~) entre la distribución de géneros
    del historial (p) y la de la lista recomendada (q), q~ = (1-a) q + a p."""
    G = _genre_matrix(items)
    G = G.div(G.sum(axis=1).clip(1e-12), axis=0)  # cada película reparte 1 entre sus géneros
    hist = train.groupby("user_id")["item_id"].agg(list).to_dict()
    vals = []
    for u, rec in recs.items():
        h = [i for i in hist.get(u, []) if i in G.index]
        rec = [i for i in rec[:k] if i in G.index]
        if not h or not rec:
            continue
        p = G.loc[h].values.mean(0)
        q = (1 - alpha) * G.loc[rec].values.mean(0) + alpha * p
        m = p > 0
        vals.append(float(np.sum(p[m] * np.log(p[m] / q[m]))))
    return float(np.mean(vals)) if vals else 0.0


def evaluate(recs: dict, test: pd.DataFrame, k: int = 10, metrics=DEFAULT_METRICS,
             rel_col: str | None = None, train: pd.DataFrame | None = None,
             items: pd.DataFrame | None = None, **kw) -> dict:
    """Media de cada métrica sobre los usuarios de test. Si se pasan `train`
    (y opcionalmente `items` con columna genres) añade métricas beyond-accuracy."""
    res = evaluate_per_user(recs, test, k, metrics, rel_col, **kw).mean().to_dict()
    test_recs = {u: recs.get(u, []) for u in test["user_id"].unique()}
    if train is not None:
        catalog = train["item_id"].unique()
        res[f"coverage@{k}"] = catalog_coverage(test_recs, catalog, k)
        res[f"gini@{k}"] = gini_index(test_recs, catalog, k)
        res[f"novelty@{k}"] = novelty(test_recs, train, k)
        if items is not None:
            res[f"ild@{k}"] = intra_list_diversity(test_recs, items, k)
            res[f"miscal_kl@{k}"] = miscalibration_kl(test_recs, train, items, k)
    return {m: float(v) for m, v in res.items()}


# ----------------------------------------------------------------------------
# Incertidumbre y significancia
# ----------------------------------------------------------------------------
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42
                 ) -> tuple[float, float, float]:
    """(media, límite inferior, límite superior) por bootstrap percentil sobre usuarios."""
    v = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), (n_boot, len(v)))
    means = v[idx].mean(1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return float(v.mean()), float(lo), float(hi)

In [ ]:
%%writefile -a cinematch_eval.py


def paired_bootstrap_test(a, b, n_boot: int = 10_000, seed: int = 42) -> dict:
    """Test pareado (mismos usuarios) de H0: media(a - b) = 0.
    Devuelve diferencia media, IC 95 % y p-valor bilateral."""
    d = np.asarray(a, float) - np.asarray(b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), (n_boot, len(d)))
    boot = d[idx].mean(1)
    centered = boot - d.mean()  # distribución bajo H0
    p = float(np.mean(np.abs(centered) >= abs(d.mean())))
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return {"diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi), "p_value": p}


# ----------------------------------------------------------------------------
# Puente con ranx (verificación)
# ----------------------------------------------------------------------------
def to_ranx(recs: dict, test: pd.DataFrame, rel_col: str | None = None):
    """Convierte (recs, test) a (Qrels, Run) de ranx. El score es decreciente con
    la posición para que ranx reproduzca exactamente nuestro orden."""
    from ranx import Qrels, Run
    rel = _relevance(test, rel_col)
    qrels = {str(u): ({str(i): int(v) for i, v in r.items()} if isinstance(r, dict)
                      else {str(i): 1 for i in r}) for u, r in rel.items()}
    run = {}
    for u in rel:
        rec = list(recs.get(u, []))
        # ranx no admite consultas vacías: un ítem centinela inexistente con score 0
        run[str(u)] = {str(i): float(len(rec) - p) for p, i in enumerate(rec)} or {"__none__": 0.0}
    return Qrels(qrels), Run(run)

In [ ]:
import cinematch_eval
importlib.reload(cinematch_eval)
from cinematch_eval import *          # noqa: F401,F403  (sustituye a las funciones TODO)

tests_unitarios(cinematch_eval)
tests_parte2(cinematch_eval)

In [ ]:
def validar_contra_ranx(m) -> None:
    nombres = {"precision": "precision", "recall": "recall", "hit_rate": "hit_rate",
               "mrr": "mrr", "map": "map", "ndcg": "ndcg"}
    peor = 0.0
    for nombre, recs in RECS.items():
        q, run = m.to_ranx(recs, test)
        qg, rung = m.to_ranx(recs, test, rel_col="rating")
        for k in (5, 10, 20):
            ours = m.evaluate(recs, test, k)
            ref = rx_evaluate(q, run, [f"{v}@{k}" for v in nombres.values()])
            peor = max(peor, *(abs(ours[f"{a}@{k}"] - ref[f"{b}@{k}"]) for a, b in nombres.items()))
            g_lin = m.evaluate(recs, test, k, metrics=["ndcg"], rel_col="rating")[f"ndcg@{k}"]
            g_exp = m.evaluate(recs, test, k, metrics=["ndcg"], rel_col="rating", exp_gain=True)[f"ndcg@{k}"]
            ref_g = rx_evaluate(qg, rung, [f"ndcg@{k}", f"ndcg_burges@{k}"])
            peor = max(peor, abs(g_lin - ref_g[f"ndcg@{k}"]), abs(g_exp - ref_g[f"ndcg_burges@{k}"]))
    assert peor < 1e-9, f"diferencia con ranx = {peor:.2e}"
    print(f"✔ coincide con ranx en {len(RECS)} runs × 3 valores de k × 8 métricas (máx. dif. {peor:.1e})")

validar_contra_ranx(cinematch_eval)

In [ ]:
informe = pd.DataFrame({m: evaluate(r, test, K, train=train_full, items=items) for m, r in RECS.items()}).T
per_user = {m: evaluate_per_user(r, test, K) for m, r in RECS.items()}
ic = pd.DataFrame({m: bootstrap_ci(d[f"ndcg@{K}"]) for m, d in per_user.items()}, index=["media", "lo", "hi"]).T
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.errorbar(ic.index, ic.media, yerr=[ic.media - ic.lo, ic.hi - ic.media], fmt="o", capsize=6, color="#4C72B0")
ax.set(title=f"CineMatch · NDCG@{K} en test con IC 95 % (bootstrap)", ylabel=f"NDCG@{K}")
plt.tight_layout(); plt.show()
informe.round(4)

In [ ]:
a, b = per_user["Item-kNN"][f"ndcg@{K}"], per_user["Popularidad"][f"ndcg@{K}"]
print("Bootstrap pareado (kNN − pop):", {k: round(v, 4) for k, v in paired_bootstrap_test(a, b).items()})
print("t-test pareado: p =", f"{stats.ttest_rel(a, b).pvalue:.3g}")
q = to_ranx(RECS["Popularidad"], test)[0]
runs = []
for m in ["Popularidad", "Popularidad 30 d", "Item-kNN"]:
    run = to_ranx(RECS[m], test)[1]; run.name = m.replace(" ", "_"); runs.append(run)
print(compare(q, runs=runs, metrics=[f"ndcg@{K}", f"recall@{K}"], max_p=0.05, stat_test="student"))

**Conclusión de referencia** (los números exactos dependen de los datos): el item-kNN supera a la popularidad en NDCG@10 y la diferencia es (o no) significativa según el test pareado — **ese** es el dato que decide, no la comparación de medias a ojo. Además gana en cobertura y novedad, lo que lo convierte en un candidato claro para el siguiente A/B. La popularidad reciente no siempre bate a la global: depende de la dinámica del catálogo, y por eso se ajusta en validación.

Desde ahora, todos los módulos del curso evaluarán así:
```python
from cinematch_eval import evaluate, evaluate_per_user, bootstrap_ci, paired_bootstrap_test
```

---
## 🚀 Retos extra
1. **Rendimiento**: vectoriza `evaluate_per_user` con NumPy (matriz de aciertos usuarios × K) y compara tiempos con `ranx` en MovieLens-1M.
2. **Métricas por segmento**: añade `evaluate_by_segment(recs, test, segment_of)` (usuarios nuevos vs veteranos, cabeza vs cola de ítems). ¿Dónde gana el kNN a la popularidad?
3. **Sampled metrics con corrección**: implementa la evaluación con 100 negativos y la corrección de Krichene & Rendle (2020); compara con full ranking.
4. **Comparaciones múltiples**: añade corrección de Holm-Bonferroni a una función `compare_models(per_user_dict, baseline)`.
5. **Tests automáticos**: convierte `tests_unitarios` y `tests_parte2` en un fichero `test_cinematch_eval.py` para `pytest` e intégralo en un workflow de CI (GitHub Actions).
6. **Serendipia**: define e implementa una métrica de serendipia (relevante ∧ no recomendado por la popularidad) y añádela a `evaluate`.

## 🤔 Reflexión (producción / MLOps)
- ¿Cómo versionarías esta librería para que un cambio en una definición de métrica no invalide en silencio la comparación con experimentos antiguos?
- ¿Qué métricas de este informe pondrías como *guardrail* en el pipeline de reentreno (módulo 17), bloqueando el despliegue si empeoran?
- Si tu métrica offline favorita no correlaciona con los resultados de los A/B de los últimos 6 meses, ¿qué harías?